# Data Scraping — `nba_api` Walkthrough

This notebook is the Phase 1 (Data Scraping) entry point for the project.

## Package overview

`nba_api` is an unofficial Python wrapper around the internal JSON endpoints that power stats.nba.com. No API key is required — it just sends HTTP requests to the same URLs the website itself uses.

It has two main areas:

- **`nba_api.stats.static`** — hardcoded, offline lookup tables (e.g. `teams.get_teams()`, `players.get_players()`). No network call, no rate-limit risk. Useful for mapping team/player IDs to names.
- **`nba_api.stats.endpoints`** — live HTTP calls to stats.nba.com (e.g. `leaguegamelog`, `teamgamelog`, `boxscoretraditionalv2`). Each endpoint is a class; instantiating it makes the request, and `.get_data_frames()` returns a list of pandas DataFrames.

A few practical notes before making live calls:

- Requests can be slow or occasionally rate-limited/blocked by stats.nba.com — it's worth setting an explicit `timeout` and being prepared to retry.
- A `season` parameter is a string like `"2023-24"`, not a single year.
- `season_type_all_star` (often just called `SeasonType`) distinguishes `"Regular Season"` from `"Playoffs"`.

In [1]:
import pandas as pd
from nba_api.stats.static import teams

# Static lookup data — no network call, just confirms the package is installed
# and shows what team metadata looks like.
nba_teams = teams.get_teams()
print(f"{len(nba_teams)} teams loaded")
pd.DataFrame(nba_teams).head()

30 teams loaded


,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966


## Choosing an endpoint: `leaguegamelog`

This project's target shape (per `Claude.md`) is one row per game across the last 5 seasons. `nba_api.stats.endpoints.leaguegamelog` is the efficient way to get there: **one API call per season returns every team's game log for that season** (two rows per game — one per team). That can later be pivoted into the mirrored `_home`/`_away` row structure the project wants.

Compare that to `teamgamelog`, which only returns one team at a time — pulling a full season that way would take 30 separate calls instead of 1.

The cell below is a single smoke-test pull for one recent completed season, just to confirm the endpoint works end-to-end. It does **not** save anything to disk yet, and it is not the full 5-season loop — that's the next step once this is confirmed working.

In [2]:
from nba_api.stats.endpoints import leaguegamelog


def fetch_league_game_log(season: str, season_type: str = "Regular Season") -> pd.DataFrame:
    """Fetch one season's league-wide game log from nba_api.

    Args:
        season: Season string, e.g. "2023-24".
        season_type: "Regular Season" or "Playoffs".

    Returns:
        DataFrame with one row per team per game for that season.
    """
    # timeout guards against stats.nba.com occasionally hanging on a request
    response = leaguegamelog.LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        timeout=60,
    )
    return response.get_data_frames()[0]


# Smoke test: pull a single recent completed season only, not the full 5-season range yet.
test_df = fetch_league_game_log("2025-26")
print(test_df.shape)
test_df.head()

(2460, 29)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,240,42,...,32,39,23,7,2,20,21,109,-10,1
1,22025,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,240,38,...,31,40,29,10,4,19,27,119,10,1
2,22025,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,290,43,...,36,52,23,6,5,25,26,124,-1,1
3,22025,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,290,46,...,27,38,29,12,4,12,27,125,1,1
4,22025,1610612746,LAC,LA Clippers,0022500087,2025-10-22,LAC @ UTA,L,240,39,...,23,38,28,8,6,15,21,108,-21,1


## What's next

Once the smoke test above runs cleanly, the next pass (a separate future step) will:

1. Loop `fetch_league_game_log` over the last 5 seasons.
2. Save each season's raw output under `data/raw/` untouched (raw data is read-only per project convention).
3. Only then reshape the combined raw data into the mirrored `_home`/`_away`, one-row-per-game structure as a processing step (output goes to `data/processed/`, not `data/raw/`).

In [4]:
import os
import time

# Get a list of the last 5 seasons
seasons = ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

# Initialise folder path
folder_name = '../data/raw/'

# Create def function to scrape and save seasons
def scrape_and_save_season_logs(seasons: list[str], folder_name: str):

    for season in seasons:

        # Create file path with season name
        file_name = f"{season}_data_raw.parquet"
        file_path = os.path.join(folder_name, file_name)

        # Verify if file path exists
        if os.path.exists(file_path):
            continue

        # Run nba_api
        print(f"Fetching data for season {season}...")

        try:
            season_df = fetch_league_game_log(season)
        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")
            continue


        # Save raw output in data/raw/ folder
        season_df.to_parquet(file_path, engine='pyarrow')

        # Run delay second timer to prevent overloading api pulls
        delay_seconds = 5
        for remaining_sec in range(delay_seconds, 0, -1):
            print(f"Waiting {remaining_sec} seconds to avoid overload...", end="\r")
            time.sleep(1)

        print(f" "*50, end='\r')

    return

# Run scrape function
scrape_and_save_season_logs(seasons=seasons, folder_name=folder_name)

### Reshape the Raw Data